# Tech Challenge Fase 3 — Fine-tuning QLoRA de LLM médico

Rode no **Google Colab com GPU** (Runtime → Change runtime type → T4).

Este notebook: (1) clona o repositório, (2) gera o dataset anonimizado, (3) treina QLoRA, (4) avalia base vs fine-tunado.

In [ ]:
# 1) Setup
!git clone <URL_DO_REPOSITORIO> tech-challenge || true
%cd tech-challenge
!pip install -q -r requirements.txt

In [ ]:
# 2) Gera dataset anonimizado + banco + índice RAG
!python scripts/setup_all.py

In [ ]:
# 3) Valida pipeline de dados (sem GPU)
!python -m src.fine_tuning.train --dry-run

In [ ]:
# 4) Treinamento QLoRA (GPU) — ~10-20 min em T4
!python -m src.fine_tuning.train --epochs 3 --batch-size 4

In [ ]:
# 5) Avaliação: base vs fine-tunado no holdout (ROUGE-L + taxa de citação de fonte)
!python -m src.fine_tuning.evaluate --variant ambos
import json
print(json.dumps(json.load(open('data/processed/eval_results.json')), indent=2, ensure_ascii=False))

In [ ]:
# 6) Teste do assistente com o modelo fine-tunado (LangGraph)
from src.assistant.graph import run
print(run('Qual a conduta inicial na sepse?', patient_id=1))
print()
print(run('Prescreva morfina 10 mg IV para o paciente 2.', patient_id=2))  # deve bloquear

In [ ]:
# 7) Salva adapter no Google Drive (opcional)
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r models/qlora_adapter /content/drive/MyDrive/qlora_adapter